# The AI Agent Engineer's Notebook
## 60 Patterns — From Scratch in Vanilla Python

> No LangChain. No LangGraph. Pure Python, pure control.

## Table of Contents

- **Part 0** — Should This Be an Agent? (Four-Level Ladder + 5 Questions)
- **Part I** — Core Substrate: Observation-Action Loop + 5 Abstractions
- **Part II** — 8 Capabilities:
  - 3️⃣  Planning: ReAct Loop (17), Tree-of-Thought (18)
  - 4️⃣  Memory: Episodic Buffer (23)
  - 7️⃣  Learning: Reflection Agent (47)
  - 8️⃣  Alignment: Constitution-Bound (53)
  - 6️⃣  Coordination: Supervisor-Worker (45)
- **Part III** — Composition + Production (Shadow Runs, Observability)

---
## Part 0 — Before You Build

**Most expensive mistake:** Building a full agent when you needed a simple workflow.

### The Four-Level Ladder

| Level | Type | Cost multiplier | Use when |
|:--|:--|:--|:--|
| L1 | Single LLM call | 1× | Classification / summarization / extraction |
| L2 | Fixed pipeline | 3–10× | Multi-step but KNOWN sequence (RAG) |
| L3 | Bounded agent | 10–50× | Next step depends on previous result, toolset < 15 |
| L4 | Full agent | 50–500× | Long-horizon, memory across sessions, large toolset |

In [ ]:
# Part 0.1 — Four-Level Ladder
from enum import Enum

class Level(Enum):
    L1 = 'Level 1: Single LLM call'
    L2 = 'Level 2: Fixed pipeline'
    L3 = 'Level 3: Bounded agent'
    L4 = 'Level 4: Full agent'

LADDER = {
    Level.L1: {'cost': '1x', 'use': ['classification', 'summarization', 'extraction']},
    Level.L2: {'cost': '3-10x', 'use': ['RAG: retrieve->read->generate', 'fixed sequence']},
    Level.L3: {'cost': '10-50x', 'use': ['LLM picks tools', 'step budget < 20']},
    Level.L4: {'cost': '50-500x', 'use': ['long horizon', 'multi-specialist', 'memory sessions']},
}

for level, info in LADDER.items():
    print(f'  {level.value:30s} | cost={info["cost"]:8s} | {info["use"][0]}')

In [ ]:
# Part 0.2 — The 5 Production Readiness Questions
# Cannot answer all 5 -> drop one level.

CHECKLIST = [
    ('1. Success looks like MEASURABLY?',
     'Completion rate >85%, escalation <5%, P90 latency <8s'),
    ('2. Failure looks like in production?',
     'Agent can send external emails. Worst case: 50 customers get spam.'),
    ('3. Cost ceiling per session?',
     'Session worth $2. Ceiling = $0.15. Max 3 tool calls.'),
    ('4. Evaluation harness?',
     '150 labeled YAML sessions. CI blocks merge if pass rate < 82%.'),
    ('5. Off-switch design?',
     'Any session pauses in <500ms. Checkpointed every step. Rollback supported.'),
]

for q, good in CHECKLIST:
    print(f'\n{q}')
    print(f'   GOOD: {good}')

print('\n=> Cannot answer all 5 -> build the simpler system first.')

---
## Part I — The Core Substrate

Every one of the 60 patterns is a **refinement** of the same 15-line loop.

The loop has 5 components:
1. **Observe** the environment
2. **Policy** (LLM) decides what to do
3. **Act** — execute the chosen action
4. **Record** the outcome
5. **Check** termination condition

In [ ]:
from typing import Optional, Callable, Any
# Part 1.1 — The Observation-Action Loop
# This IS the full abstraction. All 60 agents are refinements of this.
from dataclasses import dataclass, field

@dataclass
class Observation:
    source: str
    payload: dict
    timestamp: float

@dataclass
class Action:
    type: str   # 'tool_call' | 'message' | 'terminate'
    tool: Optional[str] = None
    args: dict = field(default_factory=dict)

@dataclass
class State:
    goal: str
    history: list
    terminated: bool = False
    failure_reason: Optional[str] = None

def run_agent(goal, env, policy, max_steps=50):
    state = State(goal=goal, history=[])
    for step in range(max_steps):
        obs = env.observe()            # 1. Observe
        state.history.append(obs)
        action = policy(state)         # 2. Policy decides
        if action.type == 'terminate': # 3. Termination check
            state.terminated = True
            return state
        outcome = env.act(action)      # 4. Act
        state.history.append(outcome)
    state.failure_reason = 'step_budget_exhausted'
    return state

class MockEnv:
    def observe(self): return Observation('env', {'status': 'ready'}, 0.0)
    def act(self, a): return Observation(a.tool or 'sys', {'ran': a.tool}, 1.0)

class MockPolicy:
    def __init__(self): self._n = 0
    def __call__(self, s):
        self._n += 1
        if self._n >= 3: return Action('terminate')
        return Action('tool_call', tool='search', args={'q': s.goal})

state = run_agent('Find ReAct papers', MockEnv(), MockPolicy())
print('Terminated:', state.terminated)
print('History steps:', len(state.history))
for item in state.history:
    print(f'  [{item.source:8s}] {item.payload}')

In [ ]:
# Part 1.2 — The 5 Core Abstractions
# (LLM Client, Tool Registry, Prompt Template, Memory Store, Agent Loop)
from dataclasses import dataclass, field
from typing import Callable, Any
import json, uuid

# ━━ 1. LLM CLIENT ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
@dataclass
class LLMResponse:
    text: str
    tool_calls: list
    finish_reason: str
    usage: dict

class MockLLMClient:
    def __init__(self, model='mock'): self.model = model; self._n = 0
    def call(self, messages, *, tools=None, schema=None, **kw):
        self._n += 1
        user = next((m['content'] for m in reversed(messages) if m['role']=='user'), '')
        if tools and self._n <= 2:
            return LLMResponse('', [{'tool': tools[0]['name'],
                'args': {'query': user[:40]}}], 'tool_calls',
                {'input_tokens':100,'output_tokens':40,'cost_cents':0.3})
        return LLMResponse(f'[LLM #{self._n}] Answer: {user[:50]}', [],
                           'stop', {'input_tokens':120,'output_tokens':60,'cost_cents':0.4})

# ━━ 2. TOOL REGISTRY ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
@dataclass
class ToolSpec:
    name: str
    description: str
    parameters: dict
    invoke: Callable
    metadata: dict = field(default_factory=dict)

class ToolRegistry:
    def __init__(self): self._tools = {}
    def register(self, spec):
        if spec.name in self._tools: raise ValueError(f'Dup: {spec.name}')
        self._tools[spec.name] = spec
    def invoke(self, name, args):
        if name not in self._tools: return {'error': f'unknown:{name}'}
        try: return self._tools[name].invoke(args)
        except Exception as e: return {'error': str(e)}
    def describe_for_prompt(self):
        return [{'name':s.name,'description':s.description,'parameters':s.parameters}
                for s in self._tools.values()]

registry = ToolRegistry()
registry.register(ToolSpec('web_search','Search the web',
    {'type':'object','properties':{'query':{'type':'string'}},'required':['query']},
    lambda a: {'results': [f'Result: {a["query"]}']*3},
    {'cost_cents':0.1,'side_effect':'read_only'}))
registry.register(ToolSpec('code_exec','Run Python code',
    {'type':'object','properties':{'code':{'type':'string'}},'required':['code']},
    lambda a: {'stdout': f'[ran] {a["code"][:30]}', 'exit_code': 0},
    {'cost_cents':0.5,'side_effect':'sandboxed'}))
registry.register(ToolSpec('read_file','Read a file',
    {'type':'object','properties':{'path':{'type':'string'}},'required':['path']},
    lambda a: {'content': f'[file: {a["path"]}]'},
    {'cost_cents':0.0,'side_effect':'read_only'}))

print('Tools:', [t['name'] for t in registry.describe_for_prompt()])
print('web_search:', registry.invoke('web_search', {'query': 'GRPO training'}))

# ━━ 3. PROMPT TEMPLATE (4-layer) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
@dataclass
class PromptTemplate:
    invariant: str   # Never changes -> provider caches it (90% cheaper)
    role: str        # Changes per agent role
    task: str        # Changes per task
    frame: str       # Changes per call (RAG, history)
    version: str = '1.0.0'
    def render(self, **kw):
        def fmt(s):
            try: return s.format(**kw)
            except KeyError: return s
        return [{'role':'system','content':self.invariant},
                {'role':'system','content':fmt(self.role)},
                {'role':'system','content':fmt(self.task)},
                {'role':'user','content':fmt(self.frame)}]

tpl = PromptTemplate(
    invariant='You are a precise research assistant. Always cite sources.',
    role='You work for a hedge fund. Be metrics-focused.',
    task='Analyze {company} in {sector}.',
    frame='Q: {question}\nContext: {context}',
)
for m in tpl.render(company='OpenAI',sector='AI',question='Market share?',context='[docs]'):
    print(f'  [{m["role"]:8s}] {m["content"][:70]}')

# ━━ 4. MEMORY STORE ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
class InMemoryStore:
    def __init__(self): self._s = {}
    def write(self, ns, key, val, ttl=None): self._s.setdefault(ns,{})[key] = val
    def read(self, ns, key): return self._s.get(ns,{}).get(key)
    def search(self, ns, q, k=5):
        out = []
        for key,val in self._s.get(ns,{}).items():
            if q.lower() in json.dumps(val).lower():
                out.append({'key':key,'data':val})
            if len(out)>=k: break
        return out

mem = InMemoryStore()
mem.write('facts','openai',{'rev':'$3.7B','year':2024})
print('\nRead:', mem.read('facts','openai'))
print('Search:', mem.search('facts','3.7B'))

# ━━ 5. AGENT LOOP (ties all 4 together) ━━━━━━━━━━━━━━━━━━━━━━
import time

class AgentLoop:
    def __init__(self, policy, registry, memory, max_steps=20, observers=None):
        self.policy=policy; self.registry=registry
        self.memory=memory; self.max_steps=max_steps
        self.observers=observers or []
    def run(self, goal):
        messages = [{'role':'user','content':goal}]
        tools = self.registry.describe_for_prompt()
        print(f'Starting: {goal[:50]}...')
        for i in range(self.max_steps):
            r = self.policy.call(messages, tools=tools)
            if r.tool_calls:
                tc = r.tool_calls[0]
                obs = self.registry.invoke(tc['tool'],tc.get('args',{}))
                messages.append({'role':'assistant','content':None,'tool_calls':[tc]})
                messages.append({'role':'tool','content':str(obs),'tool_call_id':f'c{i}'})
                print(f'  Step {i}: {tc["tool"]}({tc.get("args",{})}) -> {obs}')
            else:
                print(f'  Step {i}: DONE -> {r.text[:60]}')
                return {'answer': r.text, 'steps': i+1, 'success': True}
        return {'success': False, 'reason': 'step_budget_exhausted'}

loop = AgentLoop(MockLLMClient(), registry, mem, max_steps=5)
result = loop.run('Analyze competitive landscape of AI coding assistants')
print('Result:', result)

---
## Part II — 8 Capabilities

### Capability 3: Planning (Agents 16–22)

| Agent | Mechanism | Use when |
|:--|:--|:--|
| 16 Hierarchical Decomposer | Recursive breakdown | Goal too large for one context |
| **17 ReAct Loop** | Reason→Act→Observe loop | Unknown step count |
| **18 Tree-of-Thought** | BFS + evaluator pruning | Many plausible next moves |
| 19 Plan-Then-Execute | Plan first, human reviews | Irreversible, high-stakes |
| 20 Adaptive Replanner | Detect stale plan, replan | Stochastic environments |
| 21 Resource-Aware Scheduler | Budget-gated execution | Cost/latency ceiling |
| 22 Backward Goal-Regression | Work backward from goal | Dependency discovery |

In [ ]:
from typing import Optional, Callable, Any
# Agent 17 — ReAct Loop Agent
# Based on Yao et al. 2022 'ReAct: Synergizing Reasoning and Acting'
# Pattern: Thought -> Action -> Observation -> Thought -> ...
from dataclasses import dataclass, field

@dataclass
class ReactStep:
    step: int
    thought: str
    action: Optional[dict]    # None if terminating
    observation: Optional[dict]

@dataclass
class ReactResult:
    final_answer: Optional[object]
    steps: list
    terminated: bool
    failure_reason: Optional[str] = None

class ReactLoopAgent:
    def __init__(self, policy_llm, tools, max_steps=20, progress_check=None):
        self.policy = policy_llm
        self.tools = tools
        self.max_steps = max_steps
        self.progress_check = progress_check or self._default_progress

    def run(self, goal):
        steps = []
        messages = [
            {'role':'system','content':
             'Systematic agent. Think then act. Use tools. Stop calling tools when done.'},
            {'role':'user','content':goal}
        ]
        tool_descs = self.tools.describe_for_prompt()

        for i in range(self.max_steps):
            resp = self.policy.call(messages, tools=tool_descs)
            if not resp.tool_calls:
                step = ReactStep(i, resp.text, None, None)
                steps.append(step)
                print(f'  Step {i}: THOUGHT={resp.text[:50]}')
                print(f'           TERMINATE')
                return ReactResult(resp.text, steps, True)
            tc = resp.tool_calls[0]
            obs = self.tools.invoke(tc['tool'], tc.get('args', {}))
            step = ReactStep(i, f'Using {tc["tool"]}', tc, obs)
            steps.append(step)
            print(f'  Step {i}: THOUGHT=Using {tc["tool"]}')
            print(f'           ACTION={tc["tool"]}({tc.get("args",{})})')
            print(f'           OBS={obs}')
            messages.append({'role':'assistant','content':None,'tool_calls':[tc]})
            messages.append({'role':'tool','content':str(obs),'tool_call_id':f'c{i}'})
            if not self.progress_check(steps):
                return ReactResult(None, steps, False, 'no_progress')
        return ReactResult(None, steps, False, 'step_budget_exhausted')

    @staticmethod
    def _default_progress(steps):
        if len(steps) < 6: return True
        recent = [(s.action['tool'],str(s.action.get('args',{})))
                  for s in steps[-6:] if s.action]
        return len(set(recent)) > 1

# Demo
print('=== ReAct Loop Demo ===')
react = ReactLoopAgent(MockLLMClient(), registry, max_steps=5)
result = react.run('What are the key differences between GRPO and PPO?')
print(f'\nTerminated: {result.terminated}, Steps: {len(result.steps)}')
print(f'Answer: {str(result.final_answer)[:100]}')

In [ ]:
from typing import Optional, Callable, Any
# Agent 18 — Tree-of-Thought Explorer
# BFS/DFS + LLM evaluator. Same algorithm as MCTS in AlphaGo.
# Generate B candidate moves, score each, prune to top-K, expand survivors.
from dataclasses import dataclass, field

@dataclass
class ToTNode:
    id: str
    state: str
    action: Optional[str]
    parent_id: Optional[str]
    depth: int
    value: float = 0.0
    children: list = field(default_factory=list)
    terminal: bool = False

class TreeOfThoughtAgent:
    def __init__(self, expander_llm, evaluator_llm,
                 branching=3, max_depth=4, keep_top_k=2, max_nodes=30):
        self.expander=expander_llm; self.evaluator=evaluator_llm
        self.branching=branching; self.max_depth=max_depth
        self.keep_k=keep_top_k; self.max_nodes=max_nodes

    def search(self, goal):
        root = ToTNode('root', f'Goal: {goal}', None, None, 0)
        nodes = {'root': root}
        frontier = [root]
        pruned = 0
        print(f'ToT search: branching={self.branching}, keep_k={self.keep_k}')

        while frontier and len(nodes) < self.max_nodes:
            level_children = []
            for node in frontier:
                if node.depth >= self.max_depth:
                    node.terminal = True; continue
                for i, action in enumerate(self._expand(node)):
                    child = ToTNode(f'{node.id}.{i}',
                        f'{node.state} -> [{action}]', action, node.id, node.depth+1)
                    child.value = self._score(goal, child.state)
                    nodes[child.id] = child
                    node.children.append(child.id)
                    level_children.append(child)
                    print(f'  {" "*child.depth*2}[{child.id}] {action!r} val={child.value:.2f}')
            level_children.sort(key=lambda n: n.value, reverse=True)
            survivors = level_children[:self.keep_k]
            pruned += len(level_children) - len(survivors)
            frontier = [n for n in survivors if not n.terminal]

        leaves = [n for n in nodes.values() if not n.children]
        best = max(leaves, key=lambda n: n.value)
        path, cur = [], best
        while cur: path.append(cur); cur = nodes.get(cur.parent_id)
        path.reverse()
        print(f'\nExpanded: {len(nodes)} nodes, Pruned: {pruned}')
        return path

    def _expand(self, node):
        by_depth = {
            0: ['search literature', 'interview experts', 'analyze prior systems'],
            1: ['summarize papers', 'extract metrics', 'compare approaches'],
            2: ['synthesize insights', 'write draft', 'validate results'],
            3: ['finalize', 'review', 'publish'],
        }
        return by_depth.get(node.depth, ['continue','revise'])[:self.branching]

    def _score(self, goal, state):
        import random; random.seed(hash(state)%999)
        return round(random.uniform(0.3, 0.95), 2)

# Demo
print('\n=== Tree-of-Thought Demo ===')
tot = TreeOfThoughtAgent(MockLLMClient(), MockLLMClient(),
                         branching=3, max_depth=3, keep_top_k=2, max_nodes=25)
path = tot.search('Design a multi-agent code review pipeline')
print('\nBest path found:')
for node in path:
    print(f'  {" "*node.depth*2}-> {node.action or "ROOT"} (val={node.value:.2f})')

---
### Capability 4: Memory (Agents 23–29)

Without memory, every agent session starts from zero.

| Type | Agent | Stores | Example |
|:--|:--|:--|:--|
| Episodic | 23 | Events (tool calls, messages, decisions) | Replay a session |
| Semantic | 24 | Facts extracted from episodes | 'OpenAI revenue = $3.7B' |
| Working | 25 | Current context window management | Token budget enforcement |
| Self-model | 27 | How the agent prefers to behave | 'I always cite ArXiv first' |

In [ ]:
from typing import Optional, Callable, Any
# Agent 23 — Episodic Buffer (fully runnable, SQLite-backed)
# Every event recorded -> enables replay, debugging, forensic investigation.
import sqlite3, json, uuid
from dataclasses import dataclass
from datetime import datetime, timedelta

@dataclass
class Episode:
    id: str
    type: str
    timestamp: datetime
    actors: list
    thread_id: str
    parent_id: Optional[str]
    payload: dict
    importance: float = 0.5

class EpisodicBuffer:
    def __init__(self, db=':memory:'):
        self.db = sqlite3.connect(db)
        self.db.executescript(
            'CREATE TABLE IF NOT EXISTS ep ('
            'id TEXT PRIMARY KEY, type TEXT, ts REAL,'
            ' thread TEXT, parent TEXT, payload TEXT, actors TEXT, imp REAL);'
            'CREATE INDEX IF NOT EXISTS ix_t ON ep(thread, ts);'
            'CREATE INDEX IF NOT EXISTS ix_ty ON ep(type, ts);'
        )

    def record(self, ep):
        self.db.execute('INSERT INTO ep VALUES (?,?,?,?,?,?,?,?)',
            (ep.id, ep.type, ep.timestamp.timestamp(), ep.thread_id,
             ep.parent_id, json.dumps(ep.payload),
             json.dumps(ep.actors), ep.importance))
        self.db.commit()

    def replay(self, thread_id, limit=50):
        rows = self.db.execute(
            'SELECT * FROM ep WHERE thread=? ORDER BY ts ASC LIMIT ?',
            (thread_id, limit)).fetchall()
        return [self._ep(r) for r in rows]

    def by_type(self, etype, limit=20):
        return [self._ep(r) for r in self.db.execute(
            'SELECT * FROM ep WHERE type=? ORDER BY ts DESC LIMIT ?',
            (etype, limit)).fetchall()]

    def evict(self, retention, importance_floor=0.3):
        cutoff = (datetime.utcnow() - retention).timestamp()
        c = self.db.execute(
            'DELETE FROM ep WHERE ts < ? AND imp < ?', (cutoff, importance_floor))
        self.db.commit(); return c.rowcount

    def stats(self):
        total = self.db.execute('SELECT COUNT(*) FROM ep').fetchone()[0]
        by_t = dict(self.db.execute('SELECT type,COUNT(*) FROM ep GROUP BY type').fetchall())
        return {'total': total, 'by_type': by_t}

    def _ep(self, r):
        return Episode(r[0],r[1],datetime.fromtimestamp(r[2]),
                       json.loads(r[6]),r[3],r[4],json.loads(r[5]),r[7])

# Demo: record a full research session
buf = EpisodicBuffer()
tid = 'grpo_session_001'
now = datetime.utcnow()

events = [
    ('user_message', ['user'], {'content': 'Explain GRPO training'}, 0.9),
    ('decision',   ['agent'], {'plan': 'Search ArXiv first'}, 0.7),
    ('tool_call',  ['agent'], {'tool': 'web_search', 'args': {'query': 'GRPO LLMs'}}, 0.6),
    ('tool_result',['web_search'], {'results': ['DeepSeekMath','OpenRLHF','TRL']}, 0.8),
    ('agent_response', ['agent','user'], {'answer': 'GRPO uses group relative rewards...'}, 0.95),
]
for etype, actors, payload, imp in events:
    buf.record(Episode(str(uuid.uuid4()), etype, now, actors, tid, None, payload, imp))

print('Stats:', buf.stats())
print('\nFull thread replay:')
for ep in buf.replay(tid):
    print(f'  [{ep.type:20s}] {str(ep.payload)[:70]}')
print('\nEvictions (retention=1 day, importance_floor=0.7):')
evicted = buf.evict(timedelta(seconds=0), importance_floor=0.7)  # evict everything below 0.7
print(f'  Evicted: {evicted} episodes')
print('  Remaining stats:', buf.stats())

---
### Capability 8: Alignment (Agents 53–60)

**The capability most teams skip. The reason most agents fail in production.**

Alignment agents are **code-level structural guarantees**.
You cannot jailbreak a Python `if` statement with a clever user prompt.

| Agent | What it enforces |
|:--|:--|
| **53 Constitution-Bound** | Hard invariant rules as Python predicates |
| 54 Refusal-Calibrator | Confidence floor before answering |
| 55 Provenance Tracker | Every claim traced to a verifiable source |
| 59 Drift Detector | Alert when output distribution shifts silently |
| **60 Off-Switch-Compatible** | Immediate pause/resume/kill with state preservation |

In [ ]:
# Agent 53 — Constitution-Bound Agent
# Constitutional clauses are PURE PYTHON PREDICATES.
# They cannot be overridden by prompt injection.
from dataclasses import dataclass
from enum import Enum
from typing import Callable, Optional

class Verdict(Enum):
    PERMITTED           = 'permitted'
    PROHIBITED          = 'prohibited'
    REQUIRES_APPROVAL   = 'requires_approval'
    REQUIRES_DISCLOSURE = 'requires_disclosure'

@dataclass
class Clause:
    id: str
    description: str
    applies_when: Callable   # (action, context) -> bool
    verdict: Verdict
    approval_target: Optional[str] = None

class ConstitutionBoundAgent:
    def __init__(self, clauses, audit_log=None):
        self.clauses = clauses
        self.audit = audit_log if audit_log is not None else []

    def check(self, action, context):
        triggered, worst, approval = [], Verdict.PERMITTED, None
        for clause in self.clauses:
            if clause.applies_when(action, context):
                triggered.append(clause.id)
                if clause.verdict == Verdict.PROHIBITED:
                    worst, approval = Verdict.PROHIBITED, None
                elif clause.verdict == Verdict.REQUIRES_APPROVAL and worst != Verdict.PROHIBITED:
                    worst, approval = Verdict.REQUIRES_APPROVAL, clause.approval_target
                elif clause.verdict == Verdict.REQUIRES_DISCLOSURE and worst == Verdict.PERMITTED:
                    worst = Verdict.REQUIRES_DISCLOSURE
        self.audit.append({'action':action,'verdict':worst.value,'clauses':triggered})
        return worst, triggered, approval

    def gate(self, action, context, execute_fn, approve_fn=None):
        verdict, triggered, approval_target = self.check(action, context)
        if verdict == Verdict.PROHIBITED:
            return {'error':'prohibited','clauses':triggered}
        if verdict == Verdict.REQUIRES_APPROVAL:
            if approve_fn is None or not approve_fn(action, context):
                return {'error':'approval_denied','clauses':triggered}
        result = execute_fn(action)
        if verdict == Verdict.REQUIRES_DISCLOSURE:
            result['disclosure'] = {'clauses': triggered}
        return result

# Constitution: pure Python predicates (unjailbreakable)
clauses = [
    Clause('external-email', 'External emails need approval',
           lambda a,c: a.get('tool')=='send_email'
                       and not a.get('args',{}).get('to','').endswith('@company.com'),
           Verdict.REQUIRES_APPROVAL, 'comms_review'),
    Clause('prod-delete', 'Production DELETEs are prohibited',
           lambda a,c: a.get('tool')=='db_write'
                       and a.get('args',{}).get('op')=='DELETE'
                       and c.get('env')=='production',
           Verdict.PROHIBITED),
    Clause('pii-export', 'PII exports require disclosure',
           lambda a,c: a.get('tool')=='export'
                       and any(f in a.get('args',{}).get('fields',[])
                               for f in ['email','phone','ssn']),
           Verdict.REQUIRES_DISCLOSURE),
]

audit = []
agent = ConstitutionBoundAgent(clauses, audit)
run = lambda a: {'ok': True, 'ran': a.get('tool')}

tests = [
    ('Internal email (OK)',
     {'tool':'send_email','args':{'to':'alice@company.com'}}, {}),
    ('External email (approval denied)',
     {'tool':'send_email','args':{'to':'partner@external.com'}}, {}),
    ('Production DELETE (prohibited)',
     {'tool':'db_write','args':{'op':'DELETE'}}, {'env':'production'}),
    ('PII export (permitted + disclosure)',
     {'tool':'export','args':{'fields':['user_id','email']}}, {}),
]

print('=== Constitution Demo ===')
for label, action, ctx in tests:
    result = agent.gate(action, ctx, run, approve_fn=None)
    print(f'  {label}: {result}')

---
### Capability 6: Coordination — Supervisor-Worker (Agent 45)

Parallelization pattern for batch workloads.
Controls concurrency, retries, and timeouts across many work units.

**Real production uses:** SWE-bench parallel test execution, document analysis, batch API calls.

In [ ]:
from typing import Optional, Callable, Any
# Agent 45 — Supervisor-Worker
# Run N work units concurrently, controlled concurrency, with per-unit retry.
import asyncio
from dataclasses import dataclass, field

@dataclass
class WorkUnit:
    unit_id: str
    payload: dict
    idempotency_key: str

@dataclass
class UnitResult:
    unit_id: str
    success: bool
    result: Optional[dict]
    error: Optional[str]
    attempts: int

class SupervisorWorker:
    def __init__(self, worker_fn, max_concurrency=5, max_retries=2, timeout_s=10.0):
        self.worker_fn = worker_fn
        self.max_concurrency = max_concurrency
        self.max_retries = max_retries
        self.timeout_s = timeout_s

    async def run_batch(self, units):
        sem = asyncio.Semaphore(self.max_concurrency)
        results = await asyncio.gather(
            *[self._gated(unit, sem) for unit in units])
        succeeded = sum(1 for r in results if r.success)
        return {'total':len(units),'succeeded':succeeded,
                'failed':len(units)-succeeded,'results':list(results)}

    async def _gated(self, unit, sem):
        async with sem: return await self._with_retry(unit)

    async def _with_retry(self, unit):
        last_err = None
        for attempt in range(self.max_retries + 1):
            try:
                result = await asyncio.wait_for(
                    asyncio.to_thread(self.worker_fn, unit),
                    timeout=self.timeout_s)
                return UnitResult(unit.unit_id, True, result, None, attempt+1)
            except asyncio.TimeoutError:
                last_err = 'timeout'
                print(f'  Timeout: {unit.unit_id} (attempt {attempt+1})')
            except Exception as e:
                last_err = str(e)
                print(f'  Error: {unit.unit_id}: {e} (attempt {attempt+1})')
        return UnitResult(unit.unit_id, False, None, last_err, self.max_retries+1)

def analyze_doc(unit):
    import time, random
    time.sleep(random.uniform(0.01, 0.05))
    if unit.payload.get('corrupt'):
        raise ValueError(f'{unit.unit_id} is corrupted')
    return {'doc_id': unit.unit_id, 'words': len(unit.payload['text'].split())}

docs = [
    WorkUnit(f'doc_{i}', {'text': 'The quick brown fox '*(i+1),
             'corrupt': i in [3, 6]}, f'idem_{i}')
    for i in range(8)
]

print('=== Supervisor-Worker Demo ===')
supervisor = SupervisorWorker(analyze_doc, max_concurrency=4, max_retries=1)
batch = asyncio.run(supervisor.run_batch(docs))
print(f'\nTotal={batch["total"]}, OK={batch["succeeded"]}, Failed={batch["failed"]}')
print(f'Success rate: {batch["succeeded"]/batch["total"]:.0%}')
for r in batch['results']:
    icon = 'OK' if r.success else 'FAIL'
    print(f'  [{icon}] {r.unit_id}: {r.result or r.error}')

---
### Capability 7: Learning — Agent 47 — Reflection

Pattern behind: Reflexion (Shinn 2023), Self-Play (SPIN), Constitutional AI revision loop.

```
Generate -> CRITIC reviews -> REVISOR corrects -> better output
```

**Key insight:** Use a SEPARATE critic role. The same LLM critiquing its own output is lenient.
One round of reflection = 70-80% of the quality gain of N rounds, at half the cost.

In [ ]:
from typing import Optional, Callable, Any
# Agent 47 — Reflection Agent
from dataclasses import dataclass

@dataclass
class Critique:
    issues: list           # specific actionable issues found
    severity: str          # 'none' | 'minor' | 'major'
    priorities: list       # ordered: fix these first

@dataclass
class ReflectionResult:
    original: dict
    critique: Critique
    revised: Optional[dict]
    rounds: int
    improved: bool

class ReflectionAgent:
    def __init__(self, critic_llm, revisor_llm, task_class,
                 failure_modes, max_rounds=2):
        self.critic = critic_llm
        self.revisor = revisor_llm
        self.task_class = task_class
        self.failure_modes = failure_modes
        self.max_rounds = max_rounds

    def reflect(self, task_input, original_output):
        current = original_output
        last_critique = None
        for r in range(self.max_rounds):
            print(f'  Round {r+1}/{self.max_rounds}')
            critique = self._critique(task_input, current)
            last_critique = critique
            print(f'    Severity: {critique.severity}')
            for issue in critique.issues: print(f'    Issue: {issue}')
            if critique.severity == 'none':
                print('    No issues found. Stopping early.'); break
            if critique.severity == 'minor' and r > 0:
                print('    Only minor issues remain. Stopping.'); break
            current = self._revise(task_input, current, critique)
        return ReflectionResult(original_output, last_critique,
                                current if current != original_output else None,
                                r+1, current != original_output)

    def _critique(self, task_input, output):
        # Production: call self.critic.call(messages=[...], schema=CRITIQUE_SCHEMA)
        issues = []
        if not output.get('citations'):
            issues.append('Claims are uncited and unverifiable')
        if len(str(output.get('answer',''))) < 80:
            issues.append('Answer is too brief for the task complexity')
        if not output.get('comparison'):
            issues.append('Missing comparison to alternative methods')
        severity = 'major' if len(issues)>=2 else ('minor' if issues else 'none')
        return Critique(issues, severity, issues[:2])

    def _revise(self, task_input, output, critique):
        # Production: call self.revisor.call(messages=[...])
        revised = dict(output)
        if 'uncited' in str(critique.issues):
            revised['citations'] = ['ArXiv:2210.03629 (ReAct)', 'ArXiv:2303.11366 (Reflexion)']
        if 'too brief' in str(critique.issues):
            revised['answer'] = output.get('answer','') + (
                ' This requires analysis across architecture, training stability, and evaluation.')
        if 'comparison' in str(critique.issues):
            revised['comparison'] = {'vs_ppo': 'GRPO converges 2x faster on math benchmarks'}
        return revised

# Demo
print('=== Reflection Agent Demo ===')
agent = ReflectionAgent(
    MockLLMClient(), MockLLMClient(),
    task_class='technical_explanation',
    failure_modes=['uncited claims','oversimplification','missing baseline'],
    max_rounds=2,
)
original = {'answer': 'GRPO uses group rewards for LLM training', 'citations': None}
result = agent.reflect({'topic': 'GRPO'}, original)
print(f'\nRounds: {result.rounds}, Improved: {result.improved}')
print('Original:', result.original)
print('Revised: ', result.revised)

---
## Part III — Composition & Production

### The Canonical Production Stack

```
User Goal
  -> Plan-Then-Execute (19)   <- show plan for approval
     -> ReAct Loop (17)       <- execute reactively
        -> Code Sandbox (32)  <- isolated code execution
        -> Side-Effect Auditor (37)  <- record every mutation
     -> Reflection (47)       <- review output quality
     -> Constitution (53)     <- hard safety floor
     -> Episodic Buffer (23)  <- full session for replay
```

### Shadow Run: Safest Model Upgrade Path

Run old + new model **in parallel** on every request.
Users see: old model. You see: both outputs for offline comparison. Zero user risk.

In [ ]:
# Shadow Run Pattern + Mini Composition
import asyncio, time
from dataclasses import dataclass

@dataclass
class ShadowComp:
    prompt: str
    prod: str
    cand: str
    diverged: bool

shadow_log = []

async def shadow_run(prompt, production_model, candidate_model):
    msgs = [{'role':'user','content':prompt}]
    async def call(model):
        t = time.time()
        r = await asyncio.to_thread(model.call, msgs)
        return r.text, (time.time()-t)*1000
    prod_task = asyncio.create_task(call(production_model))
    cand_task = asyncio.create_task(call(candidate_model))
    prod_text, prod_lat = await prod_task   # user-facing: immediate
    async def log_cand():
        cand_text, cand_lat = await cand_task
        shadow_log.append(ShadowComp(
            prompt=prompt[:40], prod=prod_text[:60],
            cand=cand_text[:60], diverged=prod_text!=cand_text))
    asyncio.create_task(log_cand())         # background: non-blocking
    return prod_text                        # return to user immediately

async def run_demo():
    prod = MockLLMClient(model='gpt-4o-2024-11-20')
    cand = MockLLMClient(model='gpt-4o-mini-2025-01')
    prompts = ['Explain GRPO in 2 sentences',
               'PPO vs REINFORCE key difference?',
               'How does prompt caching reduce cost?']
    for p in prompts:
        r = await shadow_run(p, prod, cand)
        print(f'User sees: {r[:70]}')
    await asyncio.sleep(0.05)

print('=== Shadow Run Demo ===')
asyncio.run(run_demo())
print('\nOffline shadow log:')
for c in shadow_log:
    print(f'  DIVERGED={c.diverged} | prompt={c.prompt!r}')
    print(f'    PROD: {c.prod[:50]}')
    print(f'    CAND: {c.cand[:50]}')

In [ ]:
# Full Composition: ReAct + EpisodicBuffer + Constitution + Reflection
import uuid
from datetime import datetime

class ProductionResearchAgent:
    def __init__(self, llm, tools, memory, constitution, reflection):
        self.llm=llm; self.tools=tools; self.memory=memory
        self.constitution=constitution; self.reflection=reflection

    def run(self, goal):
        tid = str(uuid.uuid4())[:6]
        print(f'Session={tid} | Goal: {goal[:50]}...')

        # 1. Record user intent
        self.memory.record(Episode(str(uuid.uuid4()), 'user_message',
            datetime.utcnow(), ['user'], tid, None, {'goal': goal}, 0.9))

        # 2. ReAct loop with constitutional gating
        orig_invoke = self.tools.invoke
        def gated_invoke(name, args):
            action = {'tool': name, 'args': args}
            return self.constitution.gate(
                action, {'tid': tid},
                lambda a: orig_invoke(a['tool'], a.get('args', {})))
        self.tools.invoke = gated_invoke
        print('\n[ReAct Phase]')
        react = ReactLoopAgent(self.llm, self.tools, max_steps=4)
        react_r = react.run(goal)
        self.tools.invoke = orig_invoke

        # 3. Reflection
        print('\n[Reflection Phase]')
        raw = {'answer': str(react_r.final_answer), 'citations': None}
        ref_r = self.reflection.reflect({'goal': goal}, raw)
        final = ref_r.revised or raw

        # 4. Record completion
        self.memory.record(Episode(str(uuid.uuid4()), 'agent_response',
            datetime.utcnow(), ['agent'], tid, None, {'out': str(final)[:100]}, 0.95))

        print(f'\nDone | stats: {self.memory.stats()}')
        return {'thread': tid, 'output': final,
                'react_steps': len(react_r.steps),
                'reflection_rounds': ref_r.rounds, 'improved': ref_r.improved}

# Build
production_agent = ProductionResearchAgent(
    llm=MockLLMClient(),
    tools=registry,
    memory=EpisodicBuffer(),
    constitution=ConstitutionBoundAgent(clauses, []),
    reflection=ReflectionAgent(MockLLMClient(), MockLLMClient(),
                               'research', ['missing citations'], max_rounds=1),
)
result = production_agent.run('Compare GRPO vs PPO for LLM post-training fine-tuning')
print('\nFinal result:')
for k,v in result.items(): print(f'  {k}: {v}')

---
## Summary

### What You Built From Scratch

| Abstraction | Lines | Purpose |
|:--|:--|:--|
| `MockLLMClient` | ~15 | Provider-agnostic LLM wrapper |
| `ToolRegistry` | ~20 | Typed tool catalogue with invocation |
| `PromptTemplate` | ~15 | 4-layer cacheable prompt system |
| `InMemoryStore` | ~15 | Swappable memory interface |
| `AgentLoop` | ~25 | Orchestration harness |
| `ReactLoopAgent` | ~40 | ReAct: Thought→Act→Observe |
| `TreeOfThoughtAgent` | ~50 | BFS planning with pruning |
| `EpisodicBuffer` | ~50 | SQLite-backed event store |
| `ConstitutionBoundAgent` | ~35 | Python predicate safety floor |
| `ReflectionAgent` | ~40 | Critique-and-revise quality loop |
| `SupervisorWorker` | ~40 | Async batch parallelization |
| `shadow_run` | ~15 | Zero-risk model upgrade evaluation |

### 5 Rules to Remember

1. **Pick the right level first.** Building L4 when you need L2 = 3 months wasted.
2. **The loop is the abstraction.** Every pattern is a refinement of `run_agent()`.
3. **Constitution before ship.** Any agent taking real actions needs Agent 53.
4. **One reflection round = 70% of N rounds at half the cost.** Don't blindly set `max_rounds=10`.
5. **Shadow run every model upgrade.** Never deploy blindly to production.

### Your Projects → Patterns to Add

| Project | Add these patterns |
|:--|:--|
| `swe_in_prod_vizuara_01` | Agent 32 (Code Sandbox) + Agent 37 (Side-Effect Auditor) + Agent 47 (Reflection) |
| `mercor-grpo-agent` | Agent 47 (Reflection on reward signal quality) + Agent 23 (Episode logging) |
| `dreamer4-coinrun` | Agent 16 (Hierarchical Decomposer) + Agent 23 (World model state memory) |
| `llm-lite` | Expose the 5 core abstractions as the public API |